In [1]:
from pathlib import Path
import random
import statistics as stats

from datasets import load_from_disk
from tokenizers import Tokenizer

In [3]:
DATASET_PATH = Path.cwd().resolve()
while DATASET_PATH.name != "miniSciBERT" and DATASET_PATH.parent != DATASET_PATH:
    DATASET_PATH = DATASET_PATH.parent

DATASET_PATH = DATASET_PATH / "data" / "raw" / "research_paper2023"
TOKENIZER_PATH = DATASET_PATH.parents[2] / "models" / "my_tokenizer" / "tokenizer.json"

ds = load_from_disk(str(DATASET_PATH))["train"]
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

print("Dataset size:", len(ds))
print("Columns:", ds.column_names)
print("Tokenizer loaded successfully.")

Dataset size: 2311491
Columns: ['title', 'abstract']
Tokenizer loaded successfully.


In [4]:
for i in range(5):
    ex = ds[i]
    text = f"{ex['title']} {ex['abstract']}".strip()
    enc = tokenizer.encode(text)

    print(f"\n--- Example {i} ---")
    print("TEXT:", text[:300], "...")
    print("TOKENS:", enc.tokens[:40])
    print("TOKEN COUNT:", len(enc.ids))


--- Example 0 ---
TEXT: Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies   A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to-leading order perturbative contributions from qua ...
TOKENS: ['calculation', 'of', 'prompt', 'diphoton', 'production', 'cross', 'sections', 'at', 'tevatron', 'and', 'lhc', 'energies', 'a', 'fully', 'differential', 'calculation', 'in', 'perturbative', 'quantum', 'chromodynamics', 'is', 'presented', 'for', 'the', 'production', 'of', 'massive', 'photon', 'pairs', 'at', 'hadron', 'colliders', '.', 'all', 'next', '-', 'to', '-', 'leading', 'order']
TOKEN COUNT: 191

--- Example 1 ---
TEXT: Sparsity-certifying Graph Decompositions   We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use
it obtain a characterization of the family of $(k,\ell)$-sparse graphs and
algorithmic solutions to a family of p

In [5]:
sample_size = 1000
indices = random.sample(range(len(ds)), sample_size)

unk_id = tokenizer.token_to_id("[UNK]")
total_tokens = 0
unk_tokens = 0

for idx in indices:
    ex = ds[idx]
    text = f"{ex['title']} {ex['abstract']}".strip()
    enc = tokenizer.encode(text)

    total_tokens += len(enc.ids)
    if unk_id is not None:
        unk_tokens += sum(1 for t in enc.ids if t == unk_id)

unk_rate = unk_tokens / total_tokens if total_tokens > 0 else 0.0

print("Sample size:", sample_size)
print("Total tokens:", total_tokens)
print("UNK tokens:", unk_tokens)
print("UNK rate:", round(unk_rate * 100, 4), "%")

Sample size: 1000
Total tokens: 198519
UNK tokens: 0
UNK rate: 0.0 %


In [6]:
lengths = []

for idx in indices:
    ex = ds[idx]
    text = f"{ex['title']} {ex['abstract']}".strip()
    enc = tokenizer.encode(text)
    lengths.append(len(enc.ids))

lengths_sorted = sorted(lengths)

print("Min length:", min(lengths_sorted))
print("Mean length:", round(stats.mean(lengths_sorted), 2))
print("Median length:", round(stats.median(lengths_sorted), 2))
print("90th percentile:", lengths_sorted[int(0.9 * len(lengths_sorted))])
print("95th percentile:", lengths_sorted[int(0.95 * len(lengths_sorted))])
print("Max length:", max(lengths_sorted))

Min length: 19
Mean length: 198.52
Median length: 188.5
90th percentile: 327
95th percentile: 365
Max length: 587


In [7]:
for i in range(3):
    ex = ds[i]
    text = f"{ex['title']} {ex['abstract']}".strip()
    enc = tokenizer.encode(text)
    decoded = tokenizer.decode(enc.ids)

    print(f"\n--- Decode Check {i} ---")
    print("ORIGINAL:", text[:250], "...")
    print("DECODED :", decoded[:250], "...")


--- Decode Check 0 ---
ORIGINAL: Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies   A fully differential calculation in perturbative quantum chromodynamics is
presented for the production of massive photon pairs at hadron colliders. All
next-to ...
DECODED : calculation of prompt diphoton production cross sections at tevatron and lhc energies a fully differential calculation in perturbative quantum chromodynamics is presented for the production of massive photon pairs at hadron colliders. all next - to - ...

--- Decode Check 1 ---
ORIGINAL: Sparsity-certifying Graph Decompositions   We describe a new algorithm, the $(k,\ell)$-pebble game with colors, and use
it obtain a characterization of the family of $(k,\ell)$-sparse graphs and
algorithmic solutions to a family of problems concernin ...
DECODED : sparsity - certifying graph decompositions we describe a new algorithm, the $ ( k, \ ell ) $ - pebble game with colors, and use it obtain a charact

In [8]:
test_words = [
    "transformer",
    "transformer-based",
    "neurodegenerative",
    "electroencephalography",
    "biomedical",
    "self-supervised",
    "gene-expression",
    "multimodal"
]

for word in test_words:
    enc = tokenizer.encode(word)
    print(f"{word:25} -> {enc.tokens}")

transformer               -> ['transformer']
transformer-based         -> ['transformer', '-', 'based']
neurodegenerative         -> ['neurodegenerative']
electroencephalography    -> ['electroencephalography']
biomedical                -> ['biomedical']
self-supervised           -> ['self', '-', 'supervised']
gene-expression           -> ['gene', '-', 'expression']
multimodal                -> ['multimodal']


In [9]:
print("\n=== Quick Interpretation Guide ===")
print("- UNK rate close to 0% is good.")
print("- Common scientific words should split into a few meaningful subwords, not many tiny pieces.")
print("- Mean/median token lengths help choose max_length for training.")
print("- Decoded text should remain readable and close to original normalized text.")


=== Quick Interpretation Guide ===
- UNK rate close to 0% is good.
- Common scientific words should split into a few meaningful subwords, not many tiny pieces.
- Mean/median token lengths help choose max_length for training.
- Decoded text should remain readable and close to original normalized text.
